# FARSI Repository Branch Diff Analysis

This notebook analyzes and compares changes between the current and default branches in the FARSI repository. It will:
- Import required libraries
- Load repository information
- Compare the current branch with the default branch
- Calculate file diffs
- Display changed files and their diffs

In [ ]:
# Import Required Libraries
import os
import subprocess
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

## Load and Explore Repository Information

This section loads repository metadata (name, owner, current branch, default branch) and displays it for reference.

In [ ]:
# Load repository metadata
def get_repo_info():
    repo_name = os.path.basename(os.getcwd())
    owner = "christopher639"
    current_branch = (
        subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
    )
    default_branch = "main"
    return repo_name, owner, current_branch, default_branch

repo_name, owner, current_branch, default_branch = get_repo_info()
display(Markdown(f"**Repository:** {repo_name}<br>**Owner:** {owner}<br>**Current branch:** {current_branch}<br>**Default branch:** {default_branch}"))

## Compare Current Branch with Default Branch

This section uses git commands to compare the current branch ('main') with the default branch ('main'), identifying any differences.

In [ ]:
# Compare current branch with default branch
def get_changed_files(current_branch, default_branch):
    if current_branch == default_branch:
        # Compare with remote if on main
        diff_cmd = ["git", "diff", "origin/main", "--name-only"]
    else:
        diff_cmd = ["git", "diff", f"origin/{default_branch}..{current_branch}", "--name-only"]
    changed_files = (
        subprocess.check_output(diff_cmd).decode().strip().split("\n")
    )
    changed_files = [f for f in changed_files if f]
    return changed_files

changed_files = get_changed_files(current_branch, default_branch)
display(Markdown(f"**Changed files:** {len(changed_files)}"))
changed_files

## Calculate File Diffs

For each changed file, calculate the diff using git diff and store the results for further analysis.

In [ ]:
# Calculate diffs for each changed file
def get_file_diffs(changed_files, current_branch, default_branch):
    diffs = {}
    for file in changed_files:
        if current_branch == default_branch:
            diff_cmd = ["git", "diff", "origin/main", "--", file]
        else:
            diff_cmd = ["git", "diff", f"origin/{default_branch}..{current_branch}", "--", file]
        try:
            diff = subprocess.check_output(diff_cmd).decode(errors="replace")
        except subprocess.CalledProcessError:
            diff = "(No diff or file missing)"
        diffs[file] = diff
    return diffs

file_diffs = get_file_diffs(changed_files, current_branch, default_branch)
file_diffs

## Display Changed Files and Diff Output

This section displays a summary of changed files and shows the diff output for each file in a readable format.

In [ ]:
# Display summary and diffs
for file, diff in file_diffs.items():
    display(Markdown(f"### {file}"))
    if diff.strip():
        display(Markdown(f'```diff\n{diff}\n```'))
    else:
        display(Markdown("No changes."))